# VANE: Detecting LLM Reasoning Failures via Geometric Trajectory Analysis

This notebook demonstrates the **VANE** (Velocity, Acceleration, and Nonlinearity Estimation) framework for predicting whether an LLM's chain-of-thought reasoning will produce a correct answer.

**Key idea:** When a language model reasons incorrectly, its hidden-state trajectory across transformer layers exhibits measurably different geometric properties — higher curvature, jerk, and geodesic deviation — compared to correct reasoning.

### Contents

1. [Setup & Imports](#1-setup)
2. [The Five VANE Metrics](#2-metrics)
3. [Running the Pipeline](#3-pipeline)
4. [Loading Pre-computed Results](#4-loading)
5. [Ablation Study](#5-ablation)
6. [Selective Prediction](#6-selective)
7. [Score Distribution Analysis](#7-distributions)
8. [3D Trajectory Visualisation](#8-trajectory)
9. [Cross-Model Comparison](#9-cross-model)

---
<a id='1-setup'></a>
## 1. Setup & Imports

In [ ]:
import sys
import os
import pickle
import json
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier

# Add project root to path
sys.path.insert(0, os.path.abspath('..'))

from vane.metrics import (
    compute_metrics,
    build_features,
    build_features_full,
    get_ablation_features,
    METRIC_GROUPS,
    ALL_GROUPS,
    ALL_WINDOWS,
    ABLATION_CONFIGS,
)

%matplotlib inline

# Paper style
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 10,
    'axes.labelsize': 10,
    'axes.titlesize': 11,
    'legend.fontsize': 9,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 120,
})

BLUE   = '#2166ac'
RED    = '#b2182b'
GREEN  = '#1b7837'
PURPLE = '#8b2fc9'
GREY   = '#bababa'

print(f'VANE metrics: {ALL_GROUPS}')
print(f'Token windows: {ALL_WINDOWS}')
print(f'Ablation configs: {list(ABLATION_CONFIGS.keys())}')

---
<a id='2-metrics'></a>
## 2. The Five VANE Metrics

Given hidden states $\mathbf{h}_\ell \in \mathbb{R}^d$ at each transformer layer $\ell = 1, \dots, L$, we define layer-wise velocity vectors:

$$\Delta_\ell = \mathbf{h}_{\ell+1} - \mathbf{h}_\ell$$

and unit tangent vectors:

$$\hat{T}_\ell = \frac{\Delta_\ell}{\|\Delta_\ell\|}$$

### The five metrics

| # | Metric | Formula | Intuition |
|:-:|:---|:---|:---|
| 1 | **Curvature** | $\kappa_\ell = \|\hat{T}_{\ell+1} - \hat{T}_\ell\|$ | Frenet–Serret curvature: rate of change of direction (arc-length normalised) |
| 2 | **Jerk** | $J_\ell = \|\Delta_{\ell+1} - \Delta_\ell\|$ | Acceleration magnitude — high values indicate chaotic trajectory |
| 3 | **Velocity** | $v_\ell = \|\Delta_\ell\|$ | Step-size magnitude between consecutive layers |
| 4 | **Geodesic Deviation** | $\mathrm{dev}_\ell = \frac{\|\mathbf{h}_\ell - \mathrm{lerp}(\mathbf{h}_0, \mathbf{h}_L, \ell/L)\|}{\|\mathbf{h}_L - \mathbf{h}_0\|}$ | Normalised distance from straight-line chord — measures path efficiency |
| 5 | **Token Coherence** | $1 - \cos(\hat{T}_\ell^{(t)},\, \bar{T}_\ell)$ | How much individual token directions disagree with the mean direction |

Each metric produces a **per-layer profile** (a 1D array of length $\approx L$). These are aggregated across generated tokens via three windows:
- **max**: worst-case instability across all generated tokens
- **mean**: average instability
- **ans**: worst-case in the final answer region (last 20 tokens)

In [ ]:
# Demonstrate the metrics on synthetic hidden states
import torch

# Simulate a "correct" trajectory (smooth, geodesic-like)
L, seq, dim = 33, 50, 128  # 32 layers + embedding, 50 tokens, 128-dim
t = torch.linspace(0, 1, L).view(L, 1, 1)
start = torch.randn(1, seq, dim)
end   = torch.randn(1, seq, dim)
smooth_states = start + t * (end - start) + 0.02 * torch.randn(L, seq, dim)

# Simulate an "incorrect" trajectory (noisy, high curvature)
noisy_states = start + t * (end - start) + 0.3 * torch.randn(L, seq, dim)

# Convert to the format expected by compute_metrics: tuple of (1, seq, dim) tensors
hs_correct   = tuple(smooth_states[i:i+1] for i in range(L))
hs_incorrect = tuple(noisy_states[i:i+1] for i in range(L))

prompt_len = 10  # first 10 tokens are prompt

metrics_correct   = compute_metrics(hs_correct,   prompt_len)
metrics_incorrect = compute_metrics(hs_incorrect, prompt_len)

print('Metric profiles for CORRECT trajectory (smooth):')
for key in ['curv_max', 'jerk_max', 'vel_max', 'geodev_max', 'tokc_max']:
    vals = metrics_correct[key]
    print(f'  {key:>12s}: mean={vals.mean():.4f}  max={vals.max():.4f}  shape={vals.shape}')

print(f'  {"static_rep":>12s}: {metrics_correct["static_rep"]:.4f}')

print('\nMetric profiles for INCORRECT trajectory (noisy):')
for key in ['curv_max', 'jerk_max', 'vel_max', 'geodev_max', 'tokc_max']:
    vals = metrics_incorrect[key]
    print(f'  {key:>12s}: mean={vals.mean():.4f}  max={vals.max():.4f}  shape={vals.shape}')

print(f'  {"static_rep":>12s}: {metrics_incorrect["static_rep"]:.4f}')

In [ ]:
# Visualise the per-layer profiles for correct vs incorrect
fig, axes = plt.subplots(1, 5, figsize=(16, 3), sharey=False)

metric_names = ['Curvature', 'Jerk', 'Velocity', 'Geodesic Dev', 'Token Coherence']
metric_keys  = ['curv_max', 'jerk_max', 'vel_max', 'geodev_max', 'tokc_max']

for ax, name, key in zip(axes, metric_names, metric_keys):
    c_vals = metrics_correct[key]
    i_vals = metrics_incorrect[key]
    layers_c = np.linspace(0, 100, len(c_vals))
    layers_i = np.linspace(0, 100, len(i_vals))
    ax.plot(layers_c, c_vals, color=BLUE, linewidth=1.8, label='Correct')
    ax.plot(layers_i, i_vals, color=RED,  linewidth=1.8, label='Incorrect')
    ax.set_title(name, fontsize=10, fontweight='bold')
    ax.set_xlabel('Layer depth (%)')

axes[0].set_ylabel('Metric value')
axes[-1].legend(loc='upper right', frameon=False)
fig.tight_layout()
plt.show()

---
<a id='3-pipeline'></a>
## 3. Running the Pipeline

The full VANE pipeline consists of four stages:

1. **Generate** responses from the LLM (greedy decoding)
2. **Re-forward** the generated sequence to collect clean hidden states
3. **Compute** the five VANE metrics from the hidden states
4. **Train** a Hybrid classifier (GradientBoosting on VANE features) via cross-validation

Below is the core inference loop. For a GPU-equipped machine, uncomment and run:

In [ ]:
# ============================================================
# VANE Pipeline — Core Inference Loop (requires GPU)
# ============================================================
# Uncomment to run on a machine with a GPU and a downloaded model.
#
# from transformers import AutoModelForCausalLM, AutoTokenizer
# from datasets import load_dataset
# import torch
#
# MODEL_ID = 'meta-llama/Meta-Llama-3-8B-Instruct'
#
# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
# if tokenizer.pad_token is None:
#     tokenizer.pad_token = tokenizer.eos_token
# tokenizer.padding_side = 'left'
#
# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID, torch_dtype=torch.bfloat16
# ).to('cuda:0')
# model.eval()
#
# ds = load_dataset('openai/gsm8k', 'main', split='test')
# question = ds[0]['question']
#
# messages = [
#     {'role': 'system', 'content': 'You are a math reasoning assistant. Think step by step.'},
#     {'role': 'user',   'content': question},
# ]
# prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
# inputs = tokenizer(prompt, return_tensors='pt').to('cuda:0')
# prompt_len = inputs['input_ids'].shape[1]
#
# # Stage 1: Generate
# with torch.no_grad():
#     out = model.generate(
#         **inputs, max_new_tokens=1024, do_sample=False,
#         output_hidden_states=True, return_dict_in_generate=True
#     )
#
# # Stage 2: Re-forward for clean hidden states
# gen_ids = out.sequences
# with torch.no_grad():
#     fwd = model(gen_ids, output_hidden_states=True)
#
# # Stage 3: Compute VANE metrics
# metrics = compute_metrics(fwd.hidden_states, prompt_len)
#
# # Stage 4: Build feature vector (for classification)
# features = build_features_full(metrics)
# print(f'Feature vector shape: {features.shape}')
# print(f'Static representation: {metrics["static_rep"]:.4f}')

print('Pipeline code ready — uncomment to run on GPU.')

---
<a id='4-loading'></a>
## 4. Loading Pre-computed Results

We load the pre-computed VANE results for three models on GSM8K (1,319 samples each).

Each checkpoint contains per-sample dictionaries with:
- Layer-wise metric profiles (`curv_max`, `jerk_mean`, `vel_ans`, etc.)
- Scalar features (`static_rep`, `mean_log_prob`, `gen_length`)
- Labels (`is_correct`, `pred`, `gt`)

In [ ]:
# Update this path to where your results are stored
RESULTS_DIR = '/proj/berzelius-aiics-real/users/x_hodfa/geometric_instability/new_results_v2'

MODELS = {
    'Llama-3-8B':   'llama3-8b-instruct_gsm8k',
    'Gemma-3-12B':  'gemma-3-12b-it_gsm8k',
    'Ministral-8B': 'ministral-8b-instruct_gsm8k',
}

checkpoints = {}
classifiers = {}
selective_data = {}

for name, dirname in MODELS.items():
    base = os.path.join(RESULTS_DIR, dirname)
    
    with open(os.path.join(base, 'checkpoint.pkl'), 'rb') as f:
        ckpt = pickle.load(f)
    checkpoints[name] = ckpt
    
    with open(os.path.join(base, 'hybrid_clf.pkl'), 'rb') as f:
        classifiers[name] = pickle.load(f)
    
    with open(os.path.join(base, 'selective_prediction.json')) as f:
        selective_data[name] = json.load(f)
    
    n = len(ckpt)
    acc = sum(r['is_correct'] for r in ckpt) / n
    print(f'{name}: {n} samples, accuracy = {acc:.1%}, '
          f'layers = {ckpt[0]["num_layers"]}, '
          f'avg gen length = {np.mean([r["gen_length"] for r in ckpt]):.0f} tokens')

---
<a id='5-ablation'></a>
## 5. Ablation Study

We evaluate each VANE metric individually and combined, comparing against log-probability and static representation baselines. Each metric is evaluated using 5-fold cross-validated AUROC.

In [ ]:
def cv_auroc_logreg(X, y, n_splits=5):
    """5-fold CV AUROC with Logistic Regression."""
    X = np.nan_to_num(X.astype(np.float64), nan=0, posinf=1e6, neginf=-1e6)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    aucs = []
    for tr, te in skf.split(X, y):
        sc  = StandardScaler()
        Xtr = sc.fit_transform(X[tr])
        Xte = sc.transform(X[te])
        clf = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
        clf.fit(Xtr, y[tr])
        aucs.append(roc_auc_score(y[te], clf.predict_proba(Xte)[:, 1]))
    return np.mean(aucs), np.std(aucs)


def cv_auroc_gb(X, y, n_splits=5):
    """5-fold CV AUROC with GradientBoosting."""
    X = np.nan_to_num(X.astype(np.float64), nan=0, posinf=1e6, neginf=-1e6)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    aucs = []
    for tr, te in skf.split(X, y):
        sc  = StandardScaler()
        Xtr = sc.fit_transform(X[tr])
        Xte = sc.transform(X[te])
        clf = GradientBoostingClassifier(
            n_estimators=300, max_depth=4, learning_rate=0.05,
            subsample=0.8, min_samples_leaf=5, random_state=42
        )
        clf.fit(Xtr, y[tr])
        aucs.append(roc_auc_score(y[te], clf.predict_proba(Xte)[:, 1]))
    return np.mean(aucs), np.std(aucs)


# Run ablation for one model (Llama-3-8B)
model_name = 'Llama-3-8B'
ckpt = checkpoints[model_name]

print(f'\n{"="*60}')
print(f'  Ablation Study — {model_name} on GSM8K ({len(ckpt)} samples)')
print(f'{"="*60}')
print(f'{"Metric":<20} {"LogReg AUROC":>14}  {"GradBoost AUROC":>16}')
print(f'{"-"*54}')

for config_name in ABLATION_CONFIGS:
    X, y = get_ablation_features(ckpt, config_name)
    lr_mu, lr_sd = cv_auroc_logreg(X, y)
    if config_name == 'Hybrid':
        gb_mu, gb_sd = cv_auroc_gb(X, y)
        print(f'{config_name:<20} {lr_mu:.4f} \u00b1 {lr_sd:.4f}  {gb_mu:.4f} \u00b1 {gb_sd:.4f}')
    else:
        print(f'{config_name:<20} {lr_mu:.4f} \u00b1 {lr_sd:.4f}       ---')

print(f'{"="*60}')

In [ ]:
# Ablation dot plot — all three models
fig, ax = plt.subplots(figsize=(8, 5))

ablation_results = {
    'Llama-3-8B':   {'Log-Prob': 0.627, 'Static Rep': 0.590, 'Curvature': 0.750,
                     'Jerk': 0.723, 'Velocity': 0.734, 'Geodesic Dev': 0.692,
                     'Token Coherence': 0.731, 'VANE Hybrid': 0.759},
    'Gemma-3-12B':  {'Log-Prob': 0.541, 'Static Rep': 0.559, 'Curvature': 0.747,
                     'Jerk': 0.746, 'Velocity': 0.765, 'Geodesic Dev': 0.747,
                     'Token Coherence': 0.744, 'VANE Hybrid': 0.793},
    'Ministral-8B': {'Log-Prob': 0.585, 'Static Rep': 0.601, 'Curvature': 0.698,
                     'Jerk': 0.691, 'Velocity': 0.696, 'Geodesic Dev': 0.652,
                     'Token Coherence': 0.706, 'VANE Hybrid': 0.711},
}

metrics_order = ['Log-Prob', 'Static Rep', 'Curvature', 'Jerk', 'Velocity',
                 'Geodesic Dev', 'Token Coherence', 'VANE Hybrid']
colors = {'Llama-3-8B': BLUE, 'Gemma-3-12B': GREEN, 'Ministral-8B': PURPLE}
markers = {'Llama-3-8B': 'o', 'Gemma-3-12B': 's', 'Ministral-8B': 'D'}

y_positions = np.arange(len(metrics_order))

for i, (model, results) in enumerate(ablation_results.items()):
    values = [results[m] for m in metrics_order]
    offset = (i - 1) * 0.2
    ax.scatter(values, y_positions + offset, color=colors[model],
               marker=markers[model], s=70, zorder=3, label=model, edgecolors='white', linewidths=0.5)

# Highlight the Hybrid row
ax.axhspan(len(metrics_order) - 1.5, len(metrics_order) - 0.5,
           color='#f0f0f0', zorder=0)

# Chance line
ax.axvline(0.5, color=GREY, linewidth=0.8, linestyle='--', alpha=0.5, label='Chance')

ax.set_yticks(y_positions)
ax.set_yticklabels(metrics_order)
ax.set_xlabel('AUROC (5-fold CV)', fontsize=10)
ax.set_xlim(0.45, 0.85)
ax.invert_yaxis()
ax.legend(loc='lower right', frameon=True, fancybox=False, edgecolor='#cccccc')
ax.set_title('Per-Metric Ablation — GSM8K', fontsize=12, fontweight='bold')
ax.grid(axis='x', alpha=0.2)

fig.tight_layout()
plt.show()

---
<a id='6-selective'></a>
## 6. Selective Prediction

Selective prediction allows a model to **abstain** on uncertain samples. By ranking samples using VANE's P(correct) score and only answering the top-k%, we can trade **coverage** for **accuracy**.

The key result: VANE's geometric signal **monotonically improves** accuracy as coverage decreases, while log-probability selection can actually **decrease** accuracy below the baseline.

In [ ]:
fig, (ax_vane, ax_logp) = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)

model_styles = {
    'Llama-3-8B':   (BLUE,   '-',  0.688),
    'Gemma-3-12B':  (GREEN,  '--', 0.829),
    'Ministral-8B': (PURPLE, ':',  0.738),
}

for model_name, (color, ls, baseline) in model_styles.items():
    sel = selective_data[model_name]
    coverages = sorted([float(k) for k in sel.keys()], reverse=True)
    vane_acc = [sel[str(c)]['geo_acc'] * 100 for c in coverages]
    logp_acc = [sel[str(c)]['logp_acc'] * 100 for c in coverages]
    cov_pct  = [c * 100 for c in coverages]

    ax_vane.plot(cov_pct, vane_acc, color=color, linewidth=2.2, linestyle=ls,
                label=model_name, marker='o', markersize=5)
    ax_logp.plot(cov_pct, logp_acc, color=color, linewidth=2.2, linestyle=ls,
                label=model_name, marker='o', markersize=5)

    ax_vane.axhline(baseline * 100, color=color, linewidth=0.8, linestyle='--', alpha=0.3)
    ax_logp.axhline(baseline * 100, color=color, linewidth=0.8, linestyle='--', alpha=0.3)

for ax in [ax_vane, ax_logp]:
    ax.set_xlabel('Coverage (%)', fontsize=10)
    ax.set_xlim(50, 100)
    ax.invert_xaxis()

ax_vane.set_ylabel('Accuracy (%)', fontsize=10)
ax_vane.set_title('Selection by VANE score', fontsize=11, fontweight='bold')
ax_vane.set_ylim(55, 96)

ax_logp.set_title('Selection by log-probability', fontsize=11, fontweight='bold')

ax_logp.annotate(
    'Worse than\nanswering all',
    xy=(53, 62), xytext=(65, 58),
    fontsize=8.5, color=BLUE,
    arrowprops=dict(arrowstyle='->', color=BLUE, lw=1.2),
    ha='center',
)

fig.legend(*ax_vane.get_legend_handles_labels(),
           loc='lower center', ncol=3, frameon=False, fontsize=10,
           bbox_to_anchor=(0.5, -0.04))

fig.tight_layout(rect=[0, 0.06, 1, 1])
plt.show()

In [ ]:
# Coverage gain: VANE accuracy minus log-prob accuracy at each coverage level
fig, ax = plt.subplots(figsize=(7, 4))

for model_name, (color, ls, _) in model_styles.items():
    sel = selective_data[model_name]
    coverages = sorted([float(k) for k in sel.keys()], reverse=True)
    gains = [sel[str(c)]['gain'] * 100 for c in coverages]
    cov_pct = [c * 100 for c in coverages]

    ax.plot(cov_pct, gains, color=color, linewidth=2.2, linestyle=ls,
            label=model_name, marker='o', markersize=6)
    ax.fill_between(cov_pct, 0, gains, color=color, alpha=0.08)

ax.axhline(0, color='black', linewidth=0.6)
ax.set_xlabel('Coverage (%)', fontsize=10)
ax.set_ylabel('Accuracy gain (pp)', fontsize=10)
ax.set_title('VANE advantage over log-probability selection', fontsize=11, fontweight='bold')
ax.set_xlim(50, 100)
ax.invert_xaxis()
ax.legend(frameon=True, fancybox=False, edgecolor='#cccccc')
ax.grid(axis='y', alpha=0.2)

fig.tight_layout()
plt.show()

---
<a id='7-distributions'></a>
## 7. Score Distribution Analysis

We examine how VANE's P(correct) scores separate correct and incorrect samples. A well-calibrated detector should produce **bimodal** distributions with high P(correct) for correct samples and low P(correct) for incorrect ones.

In [ ]:
from scipy.stats import gaussian_kde, ks_2samp

def logit_kde(vals, x_grid, bw=0.12):
    """KDE in logit space to remove boundary bias at 0 and 1."""
    eps = 1e-4
    vals = np.clip(vals, eps, 1 - eps)
    logit_v = np.log(vals / (1 - vals))
    kde = gaussian_kde(logit_v, bw_method=bw)
    x_safe = np.clip(x_grid, eps, 1 - eps)
    logit_x = np.log(x_safe / (1 - x_safe))
    density = kde(logit_x) / (x_safe * (1 - x_safe))
    density[~np.isfinite(density)] = 0
    return density


fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=False)
x_grid = np.linspace(0.002, 0.998, 400)

auroc_values = {'Llama-3-8B': 0.759, 'Gemma-3-12B': 0.793, 'Ministral-8B': 0.711}

for ax, model_name in zip(axes, MODELS.keys()):
    ckpt = checkpoints[model_name]
    meta = classifiers[model_name]

    clf = meta['clf']
    sc  = meta['scaler']
    windows = meta.get('windows', ['max', 'mean'])
    ls_frac = meta.get('layer_start_frac', 0.0)
    le_frac = meta.get('layer_end_frac', 1.0)

    X = np.array([build_features_full(r, windows, ls_frac, le_frac) for r in ckpt])
    X = np.nan_to_num(X.astype(np.float64))
    vane_scores = clf.predict_proba(sc.transform(X))[:, 1]

    labels = np.array([r['is_correct'] for r in ckpt])
    correct   = vane_scores[labels == 1]
    incorrect = vane_scores[labels == 0]

    dens_c = logit_kde(correct,   x_grid)
    dens_i = logit_kde(incorrect, x_grid)

    overlap = np.minimum(dens_c, dens_i)
    ax.fill_between(x_grid, overlap, alpha=0.55, color='#cccccc', label='Overlap')
    ax.fill_between(x_grid, dens_c, alpha=0.2, color=BLUE)
    ax.fill_between(x_grid, dens_i, alpha=0.2, color=RED)
    ax.plot(x_grid, dens_c, color=BLUE, linewidth=1.8,
            label=f'Correct (n={len(correct)})')
    ax.plot(x_grid, dens_i, color=RED,  linewidth=1.8,
            label=f'Incorrect (n={len(incorrect)})')

    ax.axvline(float(np.median(correct)),   color=BLUE, lw=0.9, ls='--', alpha=0.7)
    ax.axvline(float(np.median(incorrect)), color=RED,  lw=0.9, ls='--', alpha=0.7)

    ks_stat, ks_p = ks_2samp(correct, incorrect)
    p_str = 'p < 0.001' if ks_p < 0.001 else f'p = {ks_p:.3f}'
    ax.text(0.04, 0.97, f'KS = {ks_stat:.3f}\n{p_str}',
            transform=ax.transAxes, fontsize=9, va='top', ha='left',
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#aaa', alpha=0.9, lw=0.6))

    y_peak = max(np.percentile(dens_c[dens_c > 0], 98),
                 np.percentile(dens_i[dens_i > 0], 98))
    ax.set_ylim(0, y_peak * 1.25)

    auroc = auroc_values[model_name]
    ax.set_title(f'{model_name}\n(AUROC = {auroc:.3f})', fontsize=11, fontweight='bold')
    ax.set_xlabel('VANE score P(correct)', fontsize=10)
    ax.set_xlim(0, 1)
    ax.set_yticks([])

axes[0].set_ylabel('Density', fontsize=10)

fig.legend(
    handles=[
        Patch(facecolor='#cccccc', edgecolor='#999', label='Overlap region'),
        Line2D([0], [0], color=BLUE, lw=1.6, label='Correct'),
        Line2D([0], [0], color=RED,  lw=1.6, label='Incorrect'),
    ],
    loc='lower center', ncol=3, frameon=False, fontsize=9.5,
    bbox_to_anchor=(0.5, -0.03),
)

fig.tight_layout(rect=[0, 0.06, 1, 0.92])
plt.show()

---
<a id='8-trajectory'></a>
## 8. 3D Trajectory Visualisation

We project the layer-wise hidden states into 3D using PCA and compare trajectories of correct vs incorrect reasoning. Correct trajectories tend to be **smooth and direct**, while incorrect ones are **winding and erratic**.

In [ ]:
# Load hidden states (mean-pooled across response tokens, per layer)
hs_path = os.path.join(RESULTS_DIR, 'llama3-8b-instruct_gsm8k', 'hidden_states.npz')

if os.path.exists(hs_path):
    data = np.load(hs_path)
    hidden = data['hidden']   # (N, n_layers, hidden_dim), float16
    labels = data['labels']   # (N,), int8
    N, n_layers, dim = hidden.shape
    print(f'Hidden states: {N} samples, {n_layers} layers, {dim}-dim')
    print(f'Correct: {(labels==1).sum()}, Incorrect: {(labels==0).sum()}')
else:
    print(f'Hidden states file not found at {hs_path}')
    print('Run: python scripts/run_experiment.py --save_hidden_states to generate it.')
    hidden = None

In [ ]:
if hidden is not None:
    # Find the most contrasting pair:
    # straightest correct vs most winding incorrect
    def compute_straightness(h):
        st = np.zeros(h.shape[0])
        for i in range(h.shape[0]):
            hs = h[i].astype(np.float32)
            chord = np.linalg.norm(hs[-1] - hs[0])
            path = sum(np.linalg.norm(hs[l+1] - hs[l]) for l in range(len(hs)-1))
            st[i] = chord / (path + 1e-9)
        return st

    st = compute_straightness(hidden)
    ci = np.where(labels == 1)[0][np.argmax(st[labels == 1])]
    ii = np.where(labels == 0)[0][np.argmin(st[labels == 0])]

    hs_c = hidden[ci].astype(np.float32)
    hs_i = hidden[ii].astype(np.float32)

    combined = np.vstack([hs_c, hs_i])
    pca = PCA(n_components=3)
    projected = pca.fit_transform(combined)
    pc_c = projected[:n_layers]
    pc_i = projected[n_layers:]
    var_exp = pca.explained_variance_ratio_ * 100

    fig = plt.figure(figsize=(7, 5))
    ax = fig.add_subplot(111, projection='3d')

    layer_frac = np.linspace(0, 1, n_layers)

    ax.plot(pc_c[:, 0], pc_c[:, 1], pc_c[:, 2],
            color=BLUE, linewidth=2.0, alpha=0.9, label='Correct', zorder=5)
    ax.scatter(pc_c[:, 0], pc_c[:, 1], pc_c[:, 2],
               c=layer_frac, cmap='Blues', s=20, alpha=0.7, edgecolors='white',
               linewidths=0.3, zorder=6)

    ax.plot(pc_i[:, 0], pc_i[:, 1], pc_i[:, 2],
            color=RED, linewidth=2.0, alpha=0.9, label='Incorrect', zorder=5)
    ax.scatter(pc_i[:, 0], pc_i[:, 1], pc_i[:, 2],
               c=layer_frac, cmap='Reds', s=20, alpha=0.7, edgecolors='white',
               linewidths=0.3, zorder=6)

    # Mark start and end
    for pc, color in [(pc_c, BLUE), (pc_i, RED)]:
        ax.scatter(*pc[0],  color=color, s=100, marker='^', edgecolors='black',
                   linewidths=0.8, zorder=10)
        ax.scatter(*pc[-1], color=color, s=100, marker='*', edgecolors='black',
                   linewidths=0.8, zorder=10)

    ax.set_xlabel(f'PC1 ({var_exp[0]:.1f}%)', fontsize=9)
    ax.set_ylabel(f'PC2 ({var_exp[1]:.1f}%)', fontsize=9)
    ax.set_zlabel(f'PC3 ({var_exp[2]:.1f}%)', fontsize=9)
    ax.set_title('Hidden-State Trajectory (Llama-3-8B, GSM8K)', fontsize=11, fontweight='bold')
    ax.legend(loc='upper left', frameon=True, fancybox=False, edgecolor='#cccccc')

    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False

    fig.tight_layout()
    plt.show()
else:
    print('Skipping trajectory plot (no hidden states available).')

---
<a id='9-cross-model'></a>
## 9. Cross-Model Comparison

We compare per-layer metric profiles across models to understand how VANE signals manifest differently in each architecture.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharey='row')

metric_pairs = [
    ('curv_max', 'Curvature'),
    ('vel_max',  'Velocity'),
    ('geodev_max', 'Geodesic Deviation'),
]

model_colors = {'Llama-3-8B': BLUE, 'Gemma-3-12B': GREEN, 'Ministral-8B': PURPLE}

for col, (key, title) in enumerate(metric_pairs):
    for row, (model_name, color) in enumerate([
        ('Llama-3-8B', BLUE), ('Gemma-3-12B', GREEN)
    ]):
        ax = axes[row, col]
        ckpt = checkpoints[model_name]

        correct_profiles   = [r[key] for r in ckpt if r['is_correct']]
        incorrect_profiles = [r[key] for r in ckpt if not r['is_correct']]

        # Average across samples
        n_layers = min(len(p) for p in correct_profiles)
        correct_avg   = np.mean([p[:n_layers] for p in correct_profiles], axis=0)
        incorrect_avg = np.mean([p[:n_layers] for p in incorrect_profiles], axis=0)
        correct_std   = np.std([p[:n_layers] for p in correct_profiles], axis=0)
        incorrect_std = np.std([p[:n_layers] for p in incorrect_profiles], axis=0)

        layers = np.linspace(0, 100, n_layers)

        ax.plot(layers, correct_avg, color=BLUE, linewidth=1.6, label='Correct')
        ax.fill_between(layers, correct_avg - correct_std, correct_avg + correct_std,
                        color=BLUE, alpha=0.12)
        ax.plot(layers, incorrect_avg, color=RED, linewidth=1.6, label='Incorrect')
        ax.fill_between(layers, incorrect_avg - incorrect_std, incorrect_avg + incorrect_std,
                        color=RED, alpha=0.12)

        if row == 0:
            ax.set_title(title, fontsize=11, fontweight='bold')
        if col == 0:
            ax.set_ylabel(f'{model_name}\nMetric value', fontsize=9)
        if row == 1:
            ax.set_xlabel('Layer depth (%)', fontsize=9)
        if row == 0 and col == 2:
            ax.legend(frameon=True, fancybox=False, edgecolor='#cccccc', fontsize=8)

fig.suptitle('Per-Layer Metric Profiles: Correct vs Incorrect',
             fontsize=13, fontweight='bold', y=1.01)
fig.tight_layout()
plt.show()

In [ ]:
# Summary table
print('\n' + '=' * 72)
print('  VANE Results Summary — GSM8K (1,319 samples per model)')
print('=' * 72)
print(f'{"Model":<16} {"Accuracy":>9} {"AUROC":>7} {"Sel@70%":>9} {"Sel@50%":>9} {"Gain@50%":>9}')
print('-' * 72)

for model_name in MODELS:
    ckpt = checkpoints[model_name]
    acc = sum(r['is_correct'] for r in ckpt) / len(ckpt)
    auroc = auroc_values[model_name]
    sel = selective_data[model_name]
    sel70 = sel['0.7']['geo_acc']
    sel50 = sel['0.5']['geo_acc']
    gain50 = sel['0.5']['gain']
    print(f'{model_name:<16} {acc:>8.1%} {auroc:>7.3f} {sel70:>8.1%} {sel50:>8.1%} {gain50:>+8.1%}')

print('=' * 72)
print('\nKey findings:')
print('  1. All five VANE metrics individually outperform log-prob and static baselines')
print('  2. The Hybrid classifier (GradBoost on all VANE features) achieves best AUROC')
print('  3. VANE selective prediction monotonically improves with lower coverage')
print('  4. Log-prob selection can decrease accuracy below baseline (anti-correlated)')
print('  5. Geometric signal is orthogonal to log-probability (\u0394AUROC \u2248 0 when combined)')

---

## References

If you use this code, please cite:

```bibtex
@inproceedings{vane2025,
  title     = {VANE: Detecting LLM Reasoning Failures via Geometric Trajectory Analysis},
  author    = {Hodfa, Ali},
  booktitle = {Findings of the Association for Computational Linguistics: EMNLP 2025},
  year      = {2025}
}
```